# V1 — BM25 Baseline (No Rerank)

**Pipeline:** BM25 sparse retrieval → top-5 passages  
**Metrics:** Recall@1, Recall@3, Recall@5, MRR@10  
**Đầu ra:** `outputs/eval/pipeline_results_v1.jsonl`, `outputs/eval/metrics_v1.csv`

In [1]:
# !pip install rank-bm25 tqdm

In [2]:
import re
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from pipeline_utils import (
    get_eval_qa, write_jsonl, build_corpus,
    is_hit, build_result_entry,
    compute_metrics, print_metrics, save_csv,
    EVAL_DIR
)

VERSION      = "v1"
RESULTS_PATH = EVAL_DIR / f"pipeline_results_{VERSION}.jsonl"
CSV_PATH     = EVAL_DIR / f"metrics_{VERSION}.csv"
TOP_N        = 50

print(f"Output: {RESULTS_PATH}")

d:\SGU\CNTT\NCKH2025_2026\ChatBot\cross-encoder\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Output: d:\SGU\CNTT\NCKH2025_2026\ChatBot\cross-encoder\notebooks\outputs\eval\pipeline_results_v1.jsonl


## 1. Load Corpus & Eval QA

In [3]:
corpus  = build_corpus()
eval_qa = get_eval_qa()   # tự tạo từ dev.jsonl nếu chưa có
print(f"Eval QA: {len(eval_qa)} queries")

Corpus: 1840 passages
Eval QA: 570 queries


## 2. Build BM25 Index

In [4]:
def tokenize(text: str):
    return re.sub(r'[^\w\s]', ' ', text.lower()).split()

print("Tokenizing corpus...")
tokenized_corpus = [tokenize(doc["passage"]) for doc in tqdm(corpus)]

bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 ready — {len(tokenized_corpus)} documents")

Tokenizing corpus...


100%|██████████| 1840/1840 [00:00<00:00, 49803.94it/s]

BM25 ready — 1840 documents


## 3. Evaluation

In [5]:
per_query = []
print(f"Evaluating {len(eval_qa)} queries...")

for item in tqdm(eval_qa, desc="BM25 V1"):
    ec = item["expected_citations"]

    scores  = bm25.get_scores(tokenize(item["query"]))
    top_ids = scores.argsort()[::-1][:TOP_N].tolist()

    rank_bi = -1
    for rank, idx in enumerate(top_ids, 1):
        if is_hit(idx, ec, corpus):
            rank_bi = rank
            break

    hits_at = {k: 1 if any(is_hit(i, ec, corpus) for i in top_ids[:k]) else 0
               for k in [1, 3, 5]}

    entry = build_result_entry(
        item=item, top_ids=top_ids, corpus=corpus,
        score_map=None, rank_bi=rank_bi, rank_ce=rank_bi,
        hits_at=hits_at
    )
    per_query.append(entry)

print(f"Done — {len(per_query)} queries")

Evaluating 570 queries...


BM25 V1: 100%|██████████| 570/570 [00:03<00:00, 171.81it/s]

Done — 570 queries


## 4. Results

In [6]:
metrics = compute_metrics(per_query)
print_metrics(metrics, version_label=VERSION.upper())

write_jsonl(RESULTS_PATH, per_query)
print(f"Pipeline results → {RESULTS_PATH} ({len(per_query)} rows) ✓")

save_csv(metrics, CSV_PATH, extra_cols={"version": VERSION, "bi_encoder": "BM25", "ce": "none"})


── V1 Results ──
  Metric            Value
  ------------------------
  Recall@1         0.5140
  Recall@3         0.7070
  Recall@5         0.7807
  MRR@10           0.6190
Pipeline results → d:\SGU\CNTT\NCKH2025_2026\ChatBot\cross-encoder\notebooks\outputs\eval\pipeline_results_v1.jsonl (570 rows) ✓
  Saved → d:\SGU\CNTT\NCKH2025_2026\ChatBot\cross-encoder\notebooks\outputs\eval\metrics_v1.csv ✓


## 5. Analysis — Miss Cases

In [7]:
miss_at1 = [r for r in per_query if r["hit@1"] == 0]
hit_at1  = [r for r in per_query if r["hit@1"] == 1]

print(f"Hit@1 : {len(hit_at1)}/{len(per_query)} ({len(hit_at1)/len(per_query)*100:.1f}%)")
print(f"Miss@1: {len(miss_at1)}/{len(per_query)}")

if miss_at1:
    sample = miss_at1[0]
    print(f"\nQuery   : {sample['query']}")
    print(f"Expected: {sample['expected_citations'][0]}")
    print(f"rank_bi : {sample['rank_bi']}")
    print("\nTop-3 BM25 retrieved:")
    for p in sample["top5_reranked"][:3]:
        print(f"  [{p['rank']}] {'HIT' if p['hit'] else 'miss'} | Điều {p['dieu']} Khoản {p['khoan']}")
        print(f"       {p['passage'][:100]}...")

Hit@1 : 293/570 (51.4%)
Miss@1: 277/570

Query   : nếu tôi nhận con nuôi ở một xã khác, tôi phải làm gì
Expected: {'van_ban': 'NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của Bộ Tư pháp', 'chuong': 'II', 'dieu': '11', 'khoan': '1', 'diem': None, 'chunk_index': 21}
rank_bi : 3

Top-3 BM25 retrieved:
  [1] miss | Điều 10 Khoản 1
       Khoản 1 Điều 10 NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong l...
  [2] miss | Điều 23 Khoản 1
       Khoản 1 Điều 23 NGHỊ ĐỊNH Quy định về phân quyền, phân cấp Cử Ngày 13.16.12025 trong lĩnh vực quản l...
  [3] HIT | Điều 11 Khoản 1
       Khoản 1 Điều 11 NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong l...
